# KYC/KYB Third-Party Vendor Compliance - Interactive Walkthrough

## Overview & Regulatory Context

**Regulation**: BSA/AML / FinCEN Customer Due Diligence (CDD) Rule  
**Regulators**: FinCEN, OCC, State Banking Authorities  
**Key Requirement**: Banks remain accountable for compliance even when using third-party vendors

### Critical Compliance Challenge
- **Vendor IP Protection**: Third-party KYC vendors (Jumio, Onfido, LexisNexis) protect their model internals
- **Bank Accountability**: "Third-party vendor decision is not a defense" - banks must maintain audit control
- **Examiner Access**: FinCEN/OCC examiners need complete audit trails without vendor cooperation

### Learning Objectives
1. Capture third-party KYC decisions with complete audit trails
2. Maintain bank-controlled compliance records independent of vendors
3. Enable regulatory examination without vendor involvement
4. Protect vendor IP while ensuring compliance accountability

## Setup & SDK Initialization

Let's start by importing the necessary modules and initializing the Briefcase AI SDK:

In [ ]:
import sys
import os
import uuid
import hashlib
import random
from datetime import datetime
from typing import Dict, Any, Optional

# Add shared module to path
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'shared'))

try:
    import backend
    from backend import briefcase_ai, DecisionSnapshot, Input, Output, SqliteBackend
    print("✓ Successfully imported Briefcase AI SDK and backend utilities")
except ImportError as e:
    print(f"[FAILED] Error importing required modules: {e}")
    print("Please ensure the shared backend module is available")

In [ ]:
# Initialize Briefcase AI SDK
try:
    briefcase_ai.init_with_config(2)  # Initialize with 2 worker threads
    print("✓ Briefcase AI SDK initialized successfully")
    
    # Get configured backend for audit trail storage
    db_backend = backend.get_backend()
    print("✓ SQLite backend configured for immutable audit storage")
    
except Exception as e:
    print(f"[FAILED] Failed to initialize SDK: {e}")

## Third-Party Vendor Configuration

We'll simulate working with a major KYC vendor (Jumio) while maintaining bank-controlled audit trails:

In [ ]:
# Configure third-party KYC vendor
vendor_config = {
    "vendor_name": "jumio",
    "vendor_rule_version": "jumio-kyc-rules-v4.2.1",
    "api_endpoint": "https://api.jumio.com/api/v1/verification",
    "timeout_seconds": 30
}

print("Third-Party KYC Vendor Configuration:")
print(f"  Vendor: {vendor_config['vendor_name'].title()}")
print(f"  Rule Version: {vendor_config['vendor_rule_version']}")
print(f"  API Endpoint: {vendor_config['api_endpoint']}")
print(f"  Timeout: {vendor_config['timeout_seconds']}s")
print("\n[SECURED] Vendor IP Protection: Model internals remain with vendor")
print("**Bank:** Bank Control: Complete audit trail captured by bank")

## Document Security & Customer Data Preparation

For security, we hash sensitive document content instead of storing raw documents in audit trails:

In [ ]:
def hash_document(document_content: str) -> str:
    """
    Hashes sensitive document content for secure audit trail storage.
    Never stores raw document content in audit records.
    """
    return hashlib.sha256(document_content.encode()).hexdigest()

# Simulate new account applicant
applicant_id = str(uuid.uuid4())
passport_content = f"PASSPORT_USA_{applicant_id}_JOHN_DOE"  # Simulated document

customer_data = {
    "applicant_id": applicant_id,
    "applicant_type": "individual",
    "document_type": "passport", 
    "document_hash": hash_document(passport_content),
    "submitted_name": "John Doe",
    "submitted_dob": "1985-03-15",
    "submitted_ein": None,  # Not applicable for individuals
    "vendor_rule_version": vendor_config["vendor_rule_version"],
    "vendor_api_endpoint": vendor_config["api_endpoint"]
}

print("Customer Data for KYC Verification:")
for key, value in customer_data.items():
    if key == "document_hash":
        print(f"  {key}: {value[:16]}... (hash for security)")
    elif value is not None:
        print(f"  {key}: {value}")

print("\n[ENCRYPTED] Security Note: Document content is hashed, not stored in plaintext")

## Third-Party Vendor KYC Processing

This simulates calling a real KYC vendor API (Jumio, Onfido, LexisNexis, etc.) and receiving their decision:

In [ ]:
def simulate_third_party_kyc_vendor(customer_data: Dict[str, Any], vendor_config: Dict[str, str]) -> Dict[str, Any]:
    """
    Simulates a third-party KYC vendor API call and response.
    In production, this would be actual API calls to vendors.
    """
    applicant_type = customer_data["applicant_type"]
    document_type = customer_data["document_type"]
    submitted_name = customer_data["submitted_name"]
    
    # Simulate vendor scoring logic
    identity_score = 0.0
    document_auth_score = 0.0
    sanctions_match = False
    
    # Individual verification
    if applicant_type == "individual":
        if document_type == "passport":
            document_auth_score = random.uniform(0.85, 0.98)
            identity_score = random.uniform(0.80, 0.95)
        elif document_type == "drivers_license":
            document_auth_score = random.uniform(0.70, 0.92)
            identity_score = random.uniform(0.75, 0.90)
    
    # Simulate sanctions screening
    high_risk_indicators = ["Petrov", "Kozlov", "Volkov", "DPRK", "Iran"]
    if any(indicator in submitted_name for indicator in high_risk_indicators):
        sanctions_match = True
        identity_score *= 0.7
    
    # Apply vendor-specific behavior
    vendor_name = vendor_config["vendor_name"]
    if vendor_name == "jumio":
        document_auth_score *= 0.95  # Jumio tends to be conservative
    
    # Make decision based on thresholds
    if sanctions_match:
        kyc_decision = "decline"
    elif identity_score >= 0.80 and document_auth_score >= 0.75:
        kyc_decision = "auto_approve"
    elif identity_score >= 0.60 and document_auth_score >= 0.60:
        kyc_decision = "manual_review"
    else:
        kyc_decision = "decline"
    
    return {
        "kyc_decision": kyc_decision,
        "identity_score": round(identity_score, 3),
        "document_auth_score": round(document_auth_score, 3),
        "sanctions_match_flag": sanctions_match,
        "vendor_decision_id": f"{vendor_name.upper()}-{datetime.utcnow().strftime('%Y%m%d')}-{random.randint(100000, 999999)}",
        "bank_audit_record_id": str(uuid.uuid4()),
        "vendor_rule_version": vendor_config["vendor_rule_version"],
        "vendor_api_endpoint": vendor_config["api_endpoint"]
    }

# Call the third-party vendor API
print(f"📞 Calling {vendor_config['vendor_name'].title()} KYC API...")
try:
    vendor_response = simulate_third_party_kyc_vendor(customer_data, vendor_config)
    
    print("[SUCCESS] Vendor API Response Received:")
    print(f"  KYC Decision: {vendor_response['kyc_decision']}")
    print(f"  Identity Score: {vendor_response['identity_score']}")
    print(f"  Document Auth Score: {vendor_response['document_auth_score']}")
    print(f"  Sanctions Match: {vendor_response['sanctions_match_flag']}")
    print(f"  Vendor Decision ID: {vendor_response['vendor_decision_id']}")
    print(f"  Bank Audit Record ID: {vendor_response['bank_audit_record_id']}")
    
except Exception as e:
    print(f"[FAILED] Vendor API call failed: {e}")

## Bank-Controlled Audit Trail Creation

This is the critical step: creating a complete audit trail that the bank controls, independent of the vendor:

In [ ]:
# Define regulatory metadata for FinCEN/OCC compliance
regulatory_metadata = {
    "regulation": "BSA/AML / FinCEN CDD Rule",
    "vendor_ip_exposed": False,
    "bank_controlled_audit_record": True,
    "vendor_cooperation_required_for_retrieval": False,
    "third_party_risk_documented": True,
    "vendor_name": vendor_config["vendor_name"],
    "audit_timestamp": datetime.utcnow().isoformat()
}

print("**Bank:** Creating Bank-Controlled Audit Trail...")
print("\nRegulatory Metadata:")
for key, value in regulatory_metadata.items():
    print(f"  {key}: {value}")

In [ ]:
# Create DecisionSnapshot with complete audit trail
try:
    decision_snapshot = backend.create_decision_snapshot(
        function_name="kyc_kyb_third_party_verification",
        inputs=customer_data,
        outputs=vendor_response,
        metadata=regulatory_metadata,
        input_types={
            "applicant_type": "str",
            "document_type": "str"
        },
        output_types={
            "identity_score": "float",
            "document_auth_score": "float", 
            "sanctions_match_flag": "bool"
        }
    )
    
    print("[SUCCESS] Decision snapshot created successfully")
    print(f"   Function: {decision_snapshot.function_name}")
    print(f"   Inputs captured: {len(decision_snapshot.inputs)} items")
    print(f"   Outputs captured: {len(decision_snapshot.outputs)} items")
    
except Exception as e:
    print(f"[FAILED] Error creating decision snapshot: {e}")

## Immutable Storage in Bank Systems

Store the complete audit trail in bank-controlled systems for regulatory compliance:

In [ ]:
# Store decision in bank-controlled backend
try:
    stored_decision_id = db_backend.save_decision(decision_snapshot)
    
    print("💾 Decision Stored in Bank-Controlled Audit Trail")
    print(f"   Bank Decision ID: {stored_decision_id}")
    print(f"   Bank Audit Record ID: {vendor_response['bank_audit_record_id']}")
    print(f"   Vendor Decision ID: {vendor_response['vendor_decision_id']}")
    print("\n**Objective:** Key Benefits:")
    print("   [SUCCESS] Complete audit trail under bank control")
    print("   [SUCCESS] No vendor cooperation required for retrieval")
    print("   [SUCCESS] Vendor IP protection maintained")
    print("   [SUCCESS] FinCEN examination ready")
    
except Exception as e:
    print(f"[FAILED] Error storing decision: {e}")

## Audit Trail Demonstration

Let's verify that we can retrieve the complete audit trail from our bank-controlled systems:

In [ ]:
print("="*70)
print("AUDIT TRAIL DEMONSTRATION")
print("="*70)

# Retrieve decision from bank-controlled backend
try:
    retrieved_decision = db_backend.load_decision(stored_decision_id)
    
    if retrieved_decision:
        print("[SUCCESS] Successfully retrieved complete audit trail from bank systems")
        backend.print_audit_summary(retrieved_decision)
    else:
        print("[FAILED] Failed to retrieve decision from backend")
        
except Exception as e:
    print(f"[FAILED] Error retrieving decision: {e}")

## FinCEN Examiner Query Simulation

Demonstrate how bank can respond to regulatory examination queries without vendor involvement:

In [ ]:
print("="*70)
print("FINCEN EXAMINER SIMULATION")
print("="*70)

# Simulate typical examiner query
examiner_query = f"Justify the KYC decision for account {applicant_id}. Show the complete decision process and vendor information."

print(f"**Details:** EXAMINER QUERY:")
print(f"   {examiner_query}")
print()

# Generate examiner response using bank-controlled audit trail
examiner_response = backend.format_examiner_response(
    stored_decision_id,
    examiner_query,
    db_backend
)

print("**Bank:** BANK RESPONSE (Generated from Bank-Controlled Audit Trail):")
print(examiner_response)
print()
print("**Objective:** Key Point: Response generated without any vendor cooperation or access")

## Vendor Independence Demonstration

Show how the bank maintains complete compliance capabilities even without vendor cooperation:

In [ ]:
print("="*70)
print("VENDOR INDEPENDENCE DEMONSTRATION")
print("="*70)

print("**Bank:** BANK-CONTROLLED AUDIT CAPABILITIES:")
print(f"   [SUCCESS] Complete audit record: {stored_decision_id}")
print(f"   [SUCCESS] Vendor decision ID preserved: {vendor_response['vendor_decision_id']}")
print(f"   [SUCCESS] Vendor rule version documented: {vendor_response['vendor_rule_version']}")
print(f"   [SUCCESS] Bank audit record ID: {vendor_response['bank_audit_record_id']}")
print()
print("🔓 VENDOR COOPERATION NOT REQUIRED:")
print("   [SUCCESS] Bank can retrieve complete audit trail independently")
print("   [SUCCESS] Vendor IP and internals remain protected")
print("   [SUCCESS] FinCEN examination supported without vendor involvement")
print("   [SUCCESS] Regulatory compliance maintained under bank control")

## Vendor Unavailability Scenario

Test what happens when the vendor is unavailable (system down, contract terminated, etc.):

In [ ]:
print("="*70)
print("VENDOR UNAVAILABILITY SCENARIO")
print("="*70)

print("[ALERT] SCENARIO: Vendor system is down or vendor relationship terminated")
print("**Bank:** BANK RESPONSE: Complete audit trail still available")
print()

# Retrieve decision again to demonstrate independence
try:
    independent_retrieval = db_backend.load_decision(stored_decision_id)
    
    if independent_retrieval:
        print("[SUCCESS] Audit record retrieved successfully without vendor cooperation")
        
        # Extract key information from the audit trail
        kyc_decision = independent_retrieval.outputs[0].value if independent_retrieval.outputs else 'N/A'
        vendor_used = independent_retrieval.tags.get('vendor_name', 'N/A')
        decision_timestamp = getattr(independent_retrieval, 'created_at', 'N/A')
        audit_id = getattr(independent_retrieval, 'id', 'N/A')
        
        print(f"   **Results:** KYC Decision: {kyc_decision}")
        print(f"   **Institution:** Vendor Used: {vendor_used}")
        print(f"   📅 Decision Timestamp: {decision_timestamp}")
        print(f"   🆔 Bank Audit ID: {audit_id}")
        
        print("\n**Objective:** Business Continuity Maintained:")
        print("   [SUCCESS] All regulatory data preserved")
        print("   [SUCCESS] Examination readiness unaffected")
        print("   [SUCCESS] Compliance obligations met")
        
    else:
        print("[FAILED] Failed to retrieve audit record")
        
except Exception as e:
    print(f"[FAILED] Error in independent retrieval: {e}")

## Regulatory Compliance Validation

Validate that our audit trail meets all FinCEN and BSA/AML requirements:

In [ ]:
print("="*70)
print("REGULATORY COMPLIANCE VALIDATION")
print("="*70)

# Define required fields for FinCEN KYC compliance
required_kyc_fields = [
    "regulation",
    "vendor_ip_exposed",
    "bank_controlled_audit_record",
    "vendor_cooperation_required_for_retrieval",
    "third_party_risk_documented",
    "vendor_name"
]

print("**Details:** Required FinCEN/BSA Compliance Fields:")
for field in required_kyc_fields:
    print(f"   • {field}")
print()

# Validate compliance completeness
validation_result = backend.validate_regulatory_completeness(
    retrieved_decision,
    required_kyc_fields
)

print(f"**Results:** COMPLIANCE STATUS: {'[SUCCESS] COMPLIANT' if validation_result['is_compliant'] else '[FAILED] NON-COMPLIANT'}")
print(f"**Metrics:** Completeness Score: {validation_result['completeness_score']:.1%}")

if validation_result['missing_fields']:
    print(f"[FAILED] Missing Required Fields: {', '.join(validation_result['missing_fields'])}")
else:
    print("[SUCCESS] All required compliance fields present")

print(f"\n**Achievement:** FinCEN Examination Readiness: {'READY' if validation_result['is_compliant'] else 'NOT READY'}")

## Business Continuity Benefits

Summarize the key business continuity and compliance benefits of bank-controlled audit trails:

In [ ]:
print("="*70)
print("BUSINESS CONTINUITY BENEFITS")
print("="*70)

benefits = [
    ("Vendor relationship changes", "Audit trail preserved across vendor transitions"),
    ("Vendor system outages", "Bank operations and compliance unaffected"),
    ("Regulatory examinations", "No vendor coordination required for responses"),
    ("Vendor IP protection", "Model internals never exposed to bank systems"),
    ("Multi-vendor strategies", "Consistent audit format across all vendors"),
    ("Cost management", "Reduced dependency on vendor support contracts"),
    ("Compliance confidence", "Bank maintains full control over audit quality")
]

for benefit, description in benefits:
    print(f"[SUCCESS] {benefit}: {description}")

print(f"\n**Results:** AUDIT TRAIL SUMMARY:")
print(f"   🆔 Bank Decision ID: {stored_decision_id}")
print(f"   **Institution:** Vendor Decision ID: {vendor_response['vendor_decision_id']}")
print(f"   **Details:** Bank Audit Record ID: {vendor_response['bank_audit_record_id']}")
print(f"   ⚖ Regulation: BSA/AML / FinCEN CDD Rule")
print(f"   **Bank:** Bank Controlled: [SUCCESS] Yes")
print(f"   [SECURED] Vendor IP Protected: [SUCCESS] Yes")

## Summary & Key Accomplishments

### [SUCCESS] What We Accomplished

1. **Third-Party Vendor Integration**: Successfully integrated with KYC vendor while maintaining audit independence
2. **Bank-Controlled Audit Trail**: Created complete, immutable audit records under bank control
3. **Vendor IP Protection**: Preserved vendor model confidentiality while ensuring compliance
4. **Regulatory Readiness**: Demonstrated FinCEN examination capabilities without vendor cooperation
5. **Business Continuity**: Showed resilience to vendor unavailability or relationship changes

### **Objective:** Key Compliance Benefits

- **FinCEN Accountability**: Bank maintains full responsibility and audit capabilities
- **OCC Examination Ready**: Complete audit trails available for regulatory review
- **BSA/AML Compliance**: All required decision documentation preserved
- **Third-Party Risk Management**: Vendor risk mitigated through audit independence

### **Launch:** Production Implementation Guidance

1. **API Integration**: Replace simulation with actual vendor API calls
2. **Error Handling**: Implement robust vendor API failure handling
3. **Performance**: Optimize for high-volume KYC processing
4. **Security**: Implement proper API key management and encryption
5. **Monitoring**: Set up alerts for vendor API issues and decision pattern changes

### **Reference:** Next Steps

- Integrate with your actual KYC vendor APIs
- Customize regulatory metadata for your specific jurisdiction
- Implement business logic for your risk tolerance
- Set up monitoring and alerting for compliance exceptions
- Train staff on regulatory examination response procedures

---

**[SECURED] Compliance Note**: This implementation demonstrates audit trail patterns for third-party vendor compliance. Always validate specific regulatory requirements with qualified compliance and legal counsel for your jurisdiction and business model.